In Python, a **decorator** is a design pattern that allows you to modify, extend, or supercharge the behavior of a function, method, or class **without permanently altering its actual source code.**

Syntactically, decorators are applied using the **`@`** symbol (often called "pie syntax") placed directly above a target function definition. Under the hood, they are simply higher-order functions that take a function as an input, wrap it with extra logic, and return a new, enhanced function.

---

- **🏎️ The Core Concept: Wrapper Logic**
    - To understand how a decorator works, think of a physical package. You have a core product (your function), and you place it inside a cardboard box with wrapping paper (the decorator). When someone interacts with the package, they hit the wrapping paper first before reaching the core product inside.

- **Step 1: The "Old School" Function Wrapper**
    - Because Python treats functions as first-class citizens, you can pass them around like any other object. Here is how a decorator works manually:

```python
def uppercase_decorator(func):
    # This nested function acts as the "wrapping paper"
    def wrapper():
        original_result = func()  # Execute the core function
        modified_result = original_result.upper()  # Inject custom logic
        return modified_result
    return wrapper  # Return the package

def greet():
    return "hello world"

# Manually wrapping the function
greet_extended = uppercase_decorator(greet)
print(greet_extended())  # Output: HELLO WORLD
```

- **Step 2: Clean Syntax with `@`**
    - Writing manual wrappers gets clunky quickly. Python introduces the `@` syntax to automatically pass the function through the decorator pipeline behind the scenes:

```python
@uppercase_decorator
def greet_clean():
    return "welcome home"

print(greet_clean())  # Output: WELCOME HOME
```

> By simply typing **`@uppercase_decorator`**, Python silently rewrites your code execution to match the manual wrapper pattern.

---

- **🚀 Handling Arguments with `*args` and `kwargs`**
    - The simple decorator above fails if your target function expects arguments. To make a decorator universal—capable of wrapping *any* function regardless of its signature—you must use argument unpacking inside the nested wrapper:

```python
def logger_decorator(func):
    def wrapper(*args, **kwargs):
        print(f"🎬 Calling function: '{func.__name__}'")
        result = func(*args, **kwargs)  # Captures any forwarded arguments
        print(f"✅ Finished execution.")
        return result
    return wrapper

@logger_decorator
def multiply(a, b):
    return a * b

print(multiply(4, 5))
# Output:
# 🎬 Calling function: 'multiply'
# ✅ Finished execution.
# 20
```

---

- **🛠️ Essential Real-World Utilities**
    - Decorators are incredibly powerful because they allow you to separate your core business logic from repetitive, application-wide infrastructure tasks.

- **1. Timing and Performance Benchmarking**
    - You can measure exactly how many milliseconds a process takes to execute without littering stopwatch timestamps inside every function manually.

```python
import time

def execution_timer(func):
    def wrapper(*args, **kwargs):
        start_time = time.perf_counter()
        result = func(*args, **kwargs)
        end_time = time.perf_counter()
        print(f"⏱️ '{func.__name__}' took {end_time - start_time:.4f} seconds to complete.")
        return result
    return wrapper
```

- **2. User Authentication and Permissions**
    - In web development (like Flask or Django), you can block unauthorized traffic from sensitive application endpoints using a gateway decorator.

```python
def require_login(func):
    def wrapper(user, *args, **kwargs):
        if not user.get("is_authenticated"):
            raise PermissionError("🔒 Access Denied: User is not logged in.")
        return func(user, *args, **kwargs)
    return wrapper

```

- **3. API Rate Limiting and Caching**
    - You can cache the outputs of heavy database queries or API requests so that if a user requests the exact same parameters a second time, the decorator bypasses computation entirely and serves the saved result instantly (`@functools.lru_cache` is a built-in version of this).

---

- **⚠️ Overcoming the Meta-Data Trap: `functools.wraps`**
    - When you wrap a function, its identity is hijacked by the inner `wrapper` function. If you inspect the metadata of a decorated function, you encounter an undesirable side effect:

```python
@logger_decorator
def say_hi():
    """This function greets the user."""
    return "Hi!"

print(say_hi.__name__)  # Output: wrapper (Instead of 'say_hi'!)
print(say_hi.__doc__)   # Output: None    (The docstring vanished!)
```

> This ruins debugging profiles and automated documentation engines. To fix this, Python provides **`@functools.wraps`**, a decorator designed specifically to copy the original function's name, docstring, and module traits back onto the wrapper:

```python
from functools import wraps

def polished_logger(func):
    @wraps(func)  # 🛡️ Preserves original function identity
    def wrapper(*args, **kwargs):
        return func(*args, **kwargs)
    return wrapper
```

---

- **💎 Decorator Stacking (Chaining)**
    - You can apply multiple decorators to a single function. They execute in order from **top to bottom** (or inside-out mathematically):

```python
@uppercase_decorator
@logger_decorator
def get_status():
    return "online"
```

In this scenario, `get_status` runs through the performance logger first, and then that logged output stream is passed up into the uppercase transformer.

## **Decorating functions**

In [1]:
def decorator(func):
    return func

@decorator    # Basic Implementation of Decorators
def add(a, b):
    return a + b

In [2]:
add(1, 3)

4

In [21]:
import functools

In [22]:
def decorator(func):
    @functools.wraps(func)
    def _decorator(a, b):
        # Pass the modifier argument to the function
        result = func(a, b + 3)

        # Log the function call
        name = func.__name__
        print(f"{name}(a={a}, b={b}): {result}")

        # Return a modifier output
        return result + 3

    return _decorator

@decorator
def add(a, b):
    return a + b

In [23]:
add(1, 2)

add(a=1, b=2): 6


9

### **Generic function decorators**

In [24]:
def decorator(func):
    
    @functools.wraps(func)
    def _decorator(*args, **kwargs):
        a, b = args

        return func(a, b + 3)

    return _decorator

@decorator
def add(a, b):
    return a + b

In [25]:
add(1, 2)

6

In [26]:
add(a=1, b=2)

ValueError: not enough values to unpack (expected 2, got 0)

> This code uses **positional-only arguments** (the / as the last function argument), which have been supported since Python 3.8. For older versions, you can emulate this behavior using *args instead of explicit arguments.

In [27]:
@decorator
def add(a, b, /):
    return a + b

In [28]:
add(1, 2)

6

In [29]:
add(a=1, b=2)

ValueError: not enough values to unpack (expected 2, got 0)

In [30]:
@decorator
def add(*, a, b):
    return a + b

In [31]:
add(1, 2)

TypeError: add() takes 0 positional arguments but 2 were given

In [32]:
import inspect

In [36]:
def decorator(func):
    # Use the inspect module to get function signature.
    sign = inspect.signature(func)

    @functools.wraps(func)
    def _decorator(*args, **kwargs):
        # Bind the arguments to the given *args and **kwargs.
        # If you want to make arguments optional, use
        # signature.bind_partial instead.
        bound = sign.bind(*args, **kwargs)

        # Apply the defaults so b is always filled
        bound.apply_defaults()

        # Extract the filled arguments. If the number of
        # arguments is still expected to be fixed, you can use
        # tuple unpacking: 'a, b = bound.arguments.values()'
        a = bound.arguments['a']
        b = bound.arguments['b']
        return func(a, b + 3)

    return _decorator

@decorator
def add(a, b):
    return a + b

In [37]:
add(1, b=2)

6

In [39]:
add(a=1, b=2)

6

In [43]:
add(a=1, 2)

SyntaxError: positional argument follows keyword argument (1709895177.py, line 1)

### **The importance of functools.wraps**

In [47]:
# without functionamity of functools.wraps

def decorator(func):

    def _decorator(*args, **kwargs):
        return func(*args, **kwargs)

    return _decorator

@decorator
def add(a, b):
    """Add two number a and b"""

    return a + b

In [48]:
add(1, 2)

3

In [49]:
help(add)

Help on function _decorator in module __main__:

_decorator(*args, **kwargs)



In [50]:
add.__name__

'_decorator'

In [51]:
# Adding functools.wraps to the cods

def decorator(func):

    @functools.wraps(func)
    def _decorator(*args, **kwargs):
        return func(*args, **kwargs)

    return _decorator

@decorator
def add(a, b):
    """Add two number a and b"""

    return a + b

In [52]:
add(1, 2)

3

In [53]:
help(add)

Help on function add in module __main__:

add(a, b)
    Add two number a and b



In [54]:
add.__name__

'add'

The **`functools.wraps`** utility preserves a decorated function's identity by copying and updating its core metadata attributes to the underlying wrapper function.

- Instead of using complex magic, it performs a straightforward attribute transfer, explicitly copying:
    * **`__doc__`**: The function's docstring documentation.
    * **`__name__`**: The defined name of the function.
    * **`__module__`**: The module namespace where the function was declared.
    * **`__annotations__`**: The type hints applied to inputs and outputs.
    * **`__qualname__`**: The full qualified path name of the function.

> Additionally, it syncs custom function properties by updating the wrapper’s **`__dict__`** state with the original function's dictionary contents, and injects a new **`__wrapped__`** attribute that maintains a direct reference back to the original, unmodified function object.

### **Chaining or nesting decorators**

- When stacking multiple decorators onto a single function, the execution lifecycle functions like concentric layers of an onion, making the structural order critical to keep track of. During the compilation phase, the decorators are initialized and wrapped from the **inside out** (the decorator closest to the function definition executes first). However, when the final decorated function is actually called at runtime, the processing flow triggers from the **outside in** (the outermost wrapper executes its setup logic first before passing control down the chain). Finally, once the core function completes its run, the teardown phase reverses the pipeline, processing post-execution results from the **inside back out** to the outermost layer.

In [55]:
def track(func=None, label=None):
    if label and not func:
        return functools.partial(track, label=label)

    print(f"Initializing {label}")

    @functools.wraps(func)
    def _track(*args, **kwargs):
        print(f"Calling {label}")
        func(*args, **kwargs)
        print(f"called {label}")

    return _track

In [56]:
@track(label='outer')
@track(label='inner')
def func():
    print('function')

Initializing inner
Initializing outer


In [57]:
func()

Calling outer
Calling inner
function
called inner
called outer


- This code implements a flexible decorator capable of accepting optional keyword arguments by leveraging **`functools.partial`** to freeze the **`label`** argument and return a modified decorator blueprint when a function isn't initially supplied. When stacking multiple instances of this decorator, like **`@track(label='outer')`** and **`@track(label='inner')`**, they wrap the core function in concentric layers, resembling an onion structure.

- Consequently, during the initialization and preprocessing phase, execution flows from the **outermost wrapper down to the innermost wrapper** before the actual function triggers; once the core function completes, control bubbles back upward, executing the post-processing results phase in reverse order from the **innermost wrapper back up to the outermost wrapper**.

### **Registering functions using decorators**

In [58]:
from collections import defaultdict

In [59]:
class EventsRegistry:

    def __init__(self):
        self.registry = defaultdict(list)

    def on(self, *events):
        def _on(func):
            for event in events:
                self.registry[event].append(func)
            return func

        return _on

    def fire(self, event, *args, **kwargs):
        for func in self.registry[event]:
            func(*args, **kwargs)

In [60]:
events = EventsRegistry()

@events.on('success', 'error')
def teardown(value):
    print(f"Tearing down got: {value}")

@events.on('success')
def success(value):
    print(f'Successfully executed: {value}')

In [61]:
events.fire('non-existing', 'nothing to see here')

In [62]:
events.fire('error', 'Oops, some error here')

Tearing down got: Oops, some error here


In [63]:
events.fire('success', 'Everything is fine')

Tearing down got: Everything is fine
Successfully executed: Everything is fine


### **Memoization using decorators**

In [69]:
def memoize(func):
    # Store the cache as attribute of the function so we can
    # apply the decorator to multiple functions without
    # sharing the cache.
    func.cache = dict()

    @functools.wraps(func)
    def _memoize(*args):
        # If the cache is not available, call the function
        # Note that all args need to be hashable
        if args not in func.cache:
            func.cache[args] = func(*args)
        return func.cache[args]

    return _memoize

@memoize
def fibonacci(n):
    if n < 2:
        return n
    else:
        return fibonacci(n - 1) + fibonacci(n - 2)

In [70]:
for i in range(1, 7):
    print(f"fibonnaci {i}: {fibonacci(i)}")

fibonnaci 1: 1
fibonnaci 2: 1
fibonnaci 3: 2
fibonnaci 4: 3
fibonnaci 5: 5
fibonnaci 6: 8


In [71]:
fibonacci.__wrapped__.cache

{(1,): 1, (0,): 0, (2,): 1, (3,): 2, (4,): 3, (5,): 5, (6,): 8}

- While writing a custom memoization decorator is a valuable educational exercise, manually implementing it in production is generally redundant since Python 3.2 introduced the built-in **`functools.lru_cache`** (Least Recently Used cache). This native decorator is a more sophisticated framework utility that automatically caches a function's return values based on its arguments, allowing subsequent matching requests to bypass execution and return results instantly.

- Unlike a simple custom dictionary that grows indefinitely, **`lru_cache`** manages memory efficiently by maintaining a fixed size limit (defaulting to 128 entries) and evicting the oldest, least-requested data when full; furthermore, it tracks real-time performance statistics—such as hits and misses—giving developers the necessary insights to optimize and scale the cache size dynamically.

In [78]:
# Create a simple call counting decorator
def counter(func):
    func.calls = 0

    @functools.wraps(func)
    def _counter(*args, **kwargs):
        func.calls += 1
        return func(*args, **kwargs)

    return _counter

# Create a LRU cache with size 3
@functools.lru_cache(maxsize=3)
@counter
def fibonacci(n):
    if n < 2:
        return n
    else:
        return fibonacci(n - 1) + fibonacci(n - 2)

In [79]:
fibonacci(100)

354224848179261915075

In [81]:
fibonacci.cache_info()

CacheInfo(hits=98, misses=101, maxsize=3, currsize=3)

In [83]:
fibonacci.__wrapped__.__wrapped__.calls

101

### **Decorators with (optional) arguments**

In Python, a **callable** is simply any object that you can call using a pair of parentheses `()` and optionally pass arguments into.

If you can append `()` to an object and it executes some block of code without throwing a `TypeError: '...' object is not callable`, then that object is considered a callable.

---

- **🛠️ What Objects Are Callable?**
    - Many developers assume only standard functions are callable, but Python expands this definition to several types of objects:

- **1. Built-in and Custom Functions**
    - The most obvious callables are standard functions created with the `def` keyword or anonymous functions created via `lambda`.

```python
def greet():
    return "Hello!"

# Both are callables
greet()  
(lambda x: x * 2)(5)
```

- **2. Built-in and Custom Classes**
    - When you call a class, you are telling Python to instantiate and return a new object.

```python
# Calling the built-in 'list' class creates an empty list
my_list = list() 
```

- **3. Methods**
    - Methods are simply functions that are bound to an object instance (like `string.upper()` or `list.append()`).

- **4. Custom Objects (The `__call__` Magic Method)**
    - You can turn **any normal object instance** into a callable by defining the `__call__` magic method inside its class definition. This allows an object to retain state while behaving exactly like a function.

```python
class Multiplier:
    def __init__(self, factor):
        self.factor = factor

    def __call__(self, number):
        return number * self.factor

# 1. Create the object instance
triple = Multiplier(3)

# 2. Call the object directly like a function!
print(triple(10))  # Output: 30
```

---

- **🔍 How to Check if an Object is Callable**
    - Python provides a built-in function named **`callable()`** that takes any object as an input and returns `True` if it can be called, and `False` if it cannot.

```python
print(callable(print))      # True (It's a built-in function)
print(callable(int))        # True (It's a class)
print(callable("hello"))    # False (Strings cannot be called)
```

In [88]:
def add(func=None, add_n=0):
    # function is not callable so it's probably 'add_n'
    if not callable(func):
        # Test to make sure we don't pass 'None' as 'add_n'
        if func is not None:
            add_n = func
        return functools.partial(add, add_n=add_n)

    @functools.wraps(func)
    def _add(n):
        return func(n) + add_n

    return _add

@add
def add_zero(n):
    return n

@add(add_n=1)
def add_one(n):
    return n

@add(add_n=2)
def add_two(n):
    return n

In [89]:
add_zero(8), add_one(8), add_two(8)

(8, 9, 10)

### **Creating decorators using classes**

In [90]:
class Debug(object):

    def __init__(self, func):
        self.func = func
        # functools.update_wrapper for classe
        functools.update_wrapper(self, func)

    def __call__(self, *args, **kwargs):
        output = self.func(*args, **kwargs)
        name = self.func.__name__
        print(f"{name}({args!r}, {kwargs!r}): {output!r}")
        return output

@Debug
def add(a, b=0):
    return a + b

In [91]:
add(5)

add((5,), {}): 5


5

In [92]:
add(4, 3)

add((4, 3), {}): 7


7

## **Decorating class functions**

In [93]:
def plus_one(func):
    @functools.wraps(func)
    def _plus_one(self, n, *args):
        return func(self, n + 1, *args)

    return _plus_one

class Adder(object):
    @plus_one
    def add(self, a, b=0):
        return a + b

In [94]:
adder = Adder()

adder.add(0)

1

In [95]:
adder.add(3, 4)

8

- When a decorator is applied to a method inside a class rather than a standalone function, the underlying execution mechanism remains completely unchanged, with the only distinction being how object instances are tracked. Because methods are technically functions bound to an object, the decorator's inner wrapper function automatically receives the instance reference—conventionally named **`self`**—as its absolute first positional argument.

- This means your decorator can inspect or modify the parent object's internal properties dynamically at runtime, ensuring that wrapping class methods integrates seamlessly with standard Object-Oriented patterns without requiring any specialized syntax modifications.

### **Skipping the instance – classmethod and staticmethod**

The fundamental difference between **`@classmethod`** and **`@staticmethod`** lies in **what information is automatically passed to the method** when it is called.

While both are decorators used to define methods inside a class that can be executed without instantiating an object first, they serve completely different structural purposes.

---

- **🏎️ The Quick Breakdown**
    * **`@classmethod`**: Receives the **class itself** (conventionally named `cls`) as its implicit first argument. It can access and modify class-level state.
    * **`@staticmethod`**: Receives **no implicit arguments** (neither `self` nor `cls`). It behaves exactly like a plain, isolated function that just happens to live inside the class's namespace.

---

- **🛠️ The Structural Difference in Code**

```python
class Demo:
    class_variable = "Shared Data"

    @classmethod
    def a_class_method(cls, arg1):
        # Has access to the class namespace via 'cls'
        return f"Class method called. Accessing: {cls.class_variable}"

    @staticmethod
    def a_static_method(arg1):
        # Completely isolated. Cannot access 'self' or 'cls'
        return f"Static method called with argument: {arg1}"
```

---

- **🚀 When to Use Which? (The True Intent)**

- **1. Use `@classmethod` for Factory Methods (Alternative Constructors)**
    - The absolute highest utility of a `@classmethod` is to create **alternative constructors** for your class. If your class normally accepts specific variables, but you sometimes want to instantiate it using data from a JSON payload, a CSV row, or a formatted string, you use a class method.

```python
class User:
    def __init__(self, first_name, last_name):
        self.first_name = first_name
        self.last_name = last_name

    @classmethod
    def from_string(cls, full_name_str):
        # Parses data first, then uses 'cls' to build and return a new instance
        first, last = full_name_str.split(" ")
        return cls(first, last)  # Identical to calling User(first, last)

# Standard creation
user1 = User("John", "Doe")

# Alternative creation via Class Method Factory
user2 = User.from_string("Jane Smith")
```

* *Why this rules:* If you ever inherit from this class (**`class PowerUser(User):`**), **`cls`** automatically updates to point to the subclass **`PowerUser`**, making your constructors perfectly reusable and dynamic.

- **2. Use `@staticmethod` for Isolated Helper Functions**
    - A `@staticmethod` is used when you need a utility function that is logically connected to the class topic, but doesn't actually need to read or change any properties belonging to the class or its objects.

```python
class DateValidator:
    @staticmethod
    def is_valid_date(date_str):
        # Simply evaluates an input string; doesn't care about class or object state
        if len(date_str) == 10 and "-" in date_str:
            return True
        return False

# You can use it instantly without instantiating the class
print(DateValidator.is_valid_date("2026-06-02"))  # True
```

---

- **⚖️ Summary Comparison Matrix**

| Feature | `@classmethod` | `@staticmethod` |
| --- | --- | --- |
| **Implicit First Argument** | Yes, the class object (`cls`) | No |
| **Access Class Variables?** | **Yes**, via `cls.variable_name` | No (unless you hardcode `ClassName.var`) |
| **Access Instance Properties?** | No | No |
| **Primary Use Case** | Alternative constructors / Factories | Namespace isolation for helper utilities |
| **Subclassing Behavior** | Adapts dynamically to the subclass | Remains bound to the hardcoded method logic |

In [96]:
from pprint import pprint

In [97]:
class Spam(object):
    def some_instancemethod(self, *args, **kwargs):
        pprint(locals(), width=60)

    @classmethod
    def some_classmethod(cls, *args, **kwargs):
        pprint(locals(), width=60)

    @staticmethod
    def some_staticmethod(*args, **kwargs):
        pprint(locals(), width=60)

In [98]:
spam = Spam()

In [99]:
spam.some_instancemethod(1, 2, a=3,  b=4)

{'args': (1, 2),
 'kwargs': {'a': 3, 'b': 4},
 'self': <__main__.Spam object at 0x7f98a8589400>}


In [101]:
Spam.some_instancemethod()

TypeError: Spam.some_instancemethod() missing 1 required positional argument: 'self'

In [102]:
Spam.some_instancemethod(1, 2, a=3,  b=4)

{'args': (2,), 'kwargs': {'a': 3, 'b': 4}, 'self': 1}


In [103]:
spam.some_classmethod(1, 2, a=3,  b=4)

{'args': (1, 2),
 'cls': <class '__main__.Spam'>,
 'kwargs': {'a': 3, 'b': 4}}


In [104]:
Spam.some_classmethod(1, 2, a=3,  b=4)

{'args': (1, 2),
 'cls': <class '__main__.Spam'>,
 'kwargs': {'a': 3, 'b': 4}}


In [105]:
spam.some_staticmethod()

{'args': (), 'kwargs': {}}


In [106]:
spam.some_staticmethod(1, 2, a=3,  b=4)

{'args': (1, 2), 'kwargs': {'a': 3, 'b': 4}}


In [107]:
Spam.some_staticmethod()

{'args': (), 'kwargs': {}}


In [108]:
Spam.some_staticmethod(1, 2, a=3,  b=4)

{'args': (1, 2), 'kwargs': {'a': 3, 'b': 4}}


- Before diving deeper into decorators, it is essential to understand **Python descriptors**, which are protocols that allow you to customize and hijack the default binding behavior of object attributes. Normally, accessing an attribute simply reads or writes its data directly from the object's internal dictionary; however, if a descriptor object is assigned as the value of an attribute instead, it intercepts those standard lookups.

- By defining specific magic methods—namely **`__get__()`**, **`__set__()`**, and **`__delete__()`** the descriptor acts as a gatekeeper, giving you complete programmatic control over precisely what happens, what values are calculated, and what validation checks occur whenever that attribute is accessed, modified, or removed from the instance.

In [112]:
class Spam:
    def __init__(self, spam):
        self.spam = spam

    def __get__(self, instance, cls):
        return self.spam + instance.eggs

    def __set__(self, instance, value):
        instance.eggs = value - self.spam

class Sandwich:
    spam = Spam(5)

    def __init__(self, eggs):
        self.eggs = eggs

In [113]:
sanwich = Sandwich(1)

In [114]:
sanwich.eggs

1

In [115]:
sanwich.spam

6

In [116]:
sanwich.eggs = 10
sanwich.spam

15

In [123]:
class ClassMethod(object):
    def __init__(self, method):
        self.method = method

    def __get__(self, instancen, cls):
        @functools.wraps(self.method)
        def method(*args, **kwargs):
            return self.method(cls, *args, **kwargs)

        return method


class StaticMethod(object):
    def __init__(self, method):
        self.method = method

    def __get__(self, instance, cls):
        return self.method

In [124]:
class Sandwich:
    spam = "Spam"

    def __init__(self, spam):
        self.spam = spam

    @ClassMethod
    def some_classmethod(cls, args):
        return cls.spam, args

    @StaticMethod
    def some_staticmethod(args):
        return Sandwich.spam, args

In [125]:
sandwich = Sandwich('instance')

sandwich.spam

'instance'

In [126]:
sandwich.some_classmethod('argument')

('Spam', 'argument')

In [127]:
sandwich.some_staticmethod('argument')

('Spam', 'argument')

### **Properties – Smart descriptor usage**

> Python 3.8 added **functools.cached_property**, which functions the same as property but executes only once per instance.

In [135]:
class Sandwich(object):

    def get_eggs(self):
        print("getting eggs")
        return self._eggs

    def set_eggs(self, eggs):
        print("setting eggs to %s" % eggs)
        self._eggs = eggs

    def deled_eggs(self):
        print("deleting eggs")
        del self._eggs

    eggs = property(get_eggs, set_eggs, deled_eggs)

    @property
    def spam(self):
        return self._spam

    @spam.setter
    def spam(self, spam):
        print("setting spam to %s" %spam)
        self._spam = spam

    @spam.deleter
    def spam(self):
        print("deleting spam")
        del self._spam

    @functools.cached_property
    def becon(self):
        print("getting becon")
        return 'becon!'

In [136]:
sandwich = Sandwich()

In [137]:
sandwich.eggs = 123

setting eggs to 123


In [138]:
sandwich.eggs

getting eggs


123

In [139]:
del sandwich.eggs

deleting eggs


In [140]:
sandwich.becon

getting becon


'becon!'

In [141]:
sandwich.becon

'becon!'

In [150]:
class Property(object):
    def __init__(self, fget=None, fset=None, fdel=None):
        self.fget = fget
        self.fset = fset
        self.fdel = fdel

    def __get__(self, instance , cls):
        if isinstance is None:
            return self
        elif self.fget:
            return self.fget(instance)

    def __set__(self, instance, value):
        self.fset(instance, value)

    def __delete__(self, instance):
        self.fdel(insatnce)

    def getter(self, fget):
        return Property(fget, self.fset, self.fdel)

    def setter(self, fset):
        return Property(self.fget, fset, self.fdel)

    def deleter(self, fdel):
        return Property(self.fget, self.fset, fdel)

In [151]:
class Sandwich:
    @Property
    def eggs(self):
        return self._eggs

    @eggs.setter
    def eggs(self, value):
        self._eggs = value

    @eggs.deleter
    def eggs(self):
        del self._eggs

In [152]:
sandwich = Sandwich()

In [153]:
sandwich.eggs = 5

In [154]:
sandwich.eggs

5

In [161]:
class Sandwich(object):
    def __init__(self):
        self.registry = {}

    def __getattr__(self, key):
        print("getting %r" %key)
        return self.registry.get(key, "Undefined")

    def __setattr__(self, key, value):
        if key == 'registry':
            object.__setattr__(self, key, value)
        else:
            print("Setting %r to %r" %(key, value))
            self.registry[key] = value

    def __delattr__(self, key):
        print("Deletting %r" %key)
        del self.registry[key]

In [162]:
sandwich = Sandwich()

In [163]:
sandwich.a

getting 'a'


'Undefined'

In [164]:
sandwich.a = 1

Setting 'a' to 1


In [165]:
sandwich.a

getting 'a'


1

In [166]:
del sandwich.a

Deletting 'a'


- In Python object internals, the core difference between **`__getattr__`** and **`__getattribute__`** lies in their execution triggers, which introduces critical safety implications when managing attribute access. The `__getattr__` method acts as a fallback gatekeeper, executing *only* after Python searches the object's internal `__dict__` and fails to find the requested key, which is why it ignores existing attributes.

- In contrast, **`__getattribute__` intercepts *every single attribute lookup* indiscriminately, making it far more powerful but highly volatile; if you try to reference an internal attribute like `self.registry` within it, the call triggers `__getattribute__` again, creating an infinite recursive loop that will crash your program unless you implement explicit exclusion logic. While developers rarely need to construct these descriptor mechanisms manually, they serve as the vital architectural foundation for several core Python processes, such as resolving inheritance paths via the `super()` method.